相関解析

In [28]:
import json
import sys
from pathlib import Path

import pandas as pd
from openpyxl import Workbook

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    from common.csv_loader import load_parsed_for_analysis
except Exception:
    load_parsed_for_analysis = None

OUTPUT_ROOT = PROJECT_ROOT / "data" / "export"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ID_COL_CANDIDATES = [
    "検体ID",
    "SampleID",
    "Sample ID",
    "ID",
    "SID",
    "依頼No.",
    "検体ﾊﾞｰｺｰﾄﾞ",
    "属性",
]

GROUP_COL_CANDIDATES = ["種別", "Group", "Type", "分類"]
VALUE_PREFIXES = ("処方",)


def resolve_project_root():
    root = Path.cwd()
    if not (root / "data").exists() and root.parent.exists():
        candidate = root.parent
        if (candidate / "data").exists():
            return candidate
    return root


def discover_latest_parsed_dir(parsed_root=None):
    root = resolve_project_root()
    parsed_root = Path(parsed_root) if parsed_root is not None else root / "data" / "parsed-data"

    if not parsed_root.is_absolute():
        parsed_root = root / parsed_root

    candidates = [
        p for p in parsed_root.iterdir()
        if p.is_dir() and (p / "measurement.parquet").exists() and (p / "metadata.json").exists()
    ]

    if not candidates:
        raise FileNotFoundError(f"parsed-data not found: {parsed_root}")

    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def load_parsed_data_for_notebook(parsed_dir=None):
    if load_parsed_for_analysis is None:
        raise ImportError("common.csv_loader から load_parsed_for_analysis を読み込めませんでした。")

    parsed_dir = discover_latest_parsed_dir(parsed_dir)
    measurement_df, profile_df, metadata, prescription_columns = load_parsed_for_analysis(parsed_dir)

    measurement_df = measurement_df.copy()

    if "SampleID" not in measurement_df.columns:
        if "SID" in measurement_df.columns:
            measurement_df["SampleID"] = measurement_df["SID"].astype(str)
        elif "依頼No." in measurement_df.columns:
            measurement_df["SampleID"] = measurement_df["依頼No."].astype(str)
        else:
            measurement_df["SampleID"] = measurement_df.index.astype(str)

    id_col = "SID"
    group_col_raw = pick_col(measurement_df, GROUP_COL_CANDIDATES, default=None)
    group_col = normalize_group_col(measurement_df, group_col_raw)

    value_cols = [c for c in prescription_columns if c in measurement_df.columns]
    if not value_cols:
        value_cols = detect_value_cols(measurement_df, id_col, group_col, prefixes=VALUE_PREFIXES)

    value_cols = [c for c in value_cols if c in measurement_df.columns and not str(c).endswith("_FLAG")]

    return measurement_df, profile_df, metadata, parsed_dir, id_col, group_col, value_cols


_state = {
    "xlsx_path": None,
    "source_path": None,
    "source_label": None,
    "df": None,
    "id_col": None,
    "group_col": None,
    "value_cols": None,
}


In [ ]:
import io
import re
import itertools
import math
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

import ipywidgets as widgets
from IPython.display import display, clear_output

from openpyxl import load_workbook, Workbook
from openpyxl.drawing.image import Image as XLImage


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    from common.csv_loader import load_parsed_for_analysis
    from common.analysis_utils import plot_suite
except Exception:
    load_parsed_for_analysis = None
    plot_suite = None

OUTPUT_ROOT = PROJECT_ROOT / "data" / "export"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SHEET_PLOTS = "plots"
SHEET_SUMMARY = "summary"
SHEET_GROUP_STATS = "group_stats"
SHEET_OUTLIERS = "outliers"
SHEET_SAMPLE_METRICS = "sample_metrics"
SHEET_METRIC_DEFINITIONS = "metric_definitions"

OUT_SUFFIX = "_with_QCplots"

REGRESSION_METHODS_ALL = ["OLS", "Deming", "TheilSen", "PassingBablok"]

REGRESSION_LABELS = {
    "OLS": "OLS（最小二乗）",
    "Deming": "Deming（両軸誤差）",
    "TheilSen": "Theil-Sen（ロバスト）",
    "PassingBablok": "Passing-Bablok（ノンパラメトリック）"
}


def setup_japanese_font():
    candidates = [
        "IPAexGothic", "IPAPGothic",
        "Noto Sans CJK JP", "Noto Sans JP",
        "Yu Gothic", "YuGothic",
        "Meiryo", "MS Gothic",
        "Hiragino Sans", "TakaoGothic"
    ]

    available = {f.name for f in fm.fontManager.ttflist}
    chosen = None

    for c in candidates:
        if c in available:
            chosen = c
            break

    if chosen is None:
        for name in sorted(available):
            if any(k in name for k in ["IPA", "Noto", "Gothic", "Meiryo", "Hiragino", "ゴシック", "明朝"]):
                chosen = name
                break

    if chosen:
        plt.rcParams["font.family"] = chosen

    plt.rcParams["axes.unicode_minus"] = False


setup_japanese_font()

ID_COL_CANDIDATES = [
    "検体ID",
    "SampleID",
    "Sample ID",
    "ID",
    "SID",
    "依頼No.",
    "検体ﾊﾞｰｺｰﾄﾞ",
    "属性",
]
GROUP_COL_CANDIDATES = ["種別", "Group", "Type", "分類"]
VALUE_PREFIXES = ("処方",)


def pick_col(df, candidates, default=None):
    for c in candidates:
        if c in df.columns:
            return c
    return default


def resolve_project_root():
    root = Path.cwd()
    if not (root / "data").exists() and root.parent.exists():
        candidate = root.parent
        if (candidate / "data").exists():
            return candidate
    return root


def discover_latest_parsed_dir(parsed_root=None):
    root = resolve_project_root()
    parsed_root = Path(parsed_root) if parsed_root is not None else root / "data" / "parsed-data"

    if not parsed_root.is_absolute():
        parsed_root = root / parsed_root

    candidates = [
        p for p in parsed_root.iterdir()
        if p.is_dir() and (p / "measurement.parquet").exists() and (p / "metadata.json").exists()
    ]

    if not candidates:
        raise FileNotFoundError(f"parsed-data not found: {parsed_root}")

    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def load_parsed_data_for_notebook(parsed_dir=None):
    if load_parsed_for_analysis is None:
        raise ImportError("common.csv_loader から load_parsed_for_analysis を読み込めませんでした。")

    if parsed_dir is None:
        parsed_dir = discover_latest_parsed_dir()
    else:
        parsed_dir = Path(parsed_dir)
        if not parsed_dir.is_absolute():
            parsed_dir = resolve_project_root() / parsed_dir
        if not parsed_dir.exists():
            raise FileNotFoundError(f"parsed-data not found: {parsed_dir}")

    measurement_df, profile_df, metadata, prescription_columns = load_parsed_for_analysis(parsed_dir)

    measurement_df = measurement_df.copy()
    if "SampleID" not in measurement_df.columns:
        if "SID" in measurement_df.columns:
            measurement_df["SampleID"] = measurement_df["SID"].astype(str)
        elif "依頼No." in measurement_df.columns:
            measurement_df["SampleID"] = measurement_df["依頼No."].astype(str)
        else:
            measurement_df["SampleID"] = measurement_df.index.astype(str)

    id_col = pick_col(measurement_df, ID_COL_CANDIDATES, default="SID")
    group_col_raw = pick_col(measurement_df, GROUP_COL_CANDIDATES, default=None)
    group_col = normalize_group_col(measurement_df, group_col_raw)

    value_cols = [c for c in prescription_columns if c in measurement_df.columns]
    if not value_cols:
        value_cols = detect_value_cols(measurement_df, id_col, group_col, prefixes=VALUE_PREFIXES)

    value_cols = [c for c in value_cols if c in measurement_df.columns and not str(c).endswith("_FLAG")]

    return measurement_df, profile_df, metadata, parsed_dir, id_col, group_col, value_cols


def normalize_group_col(df, group_col):
    if group_col is None:
        return None

    if group_col not in df.columns:
        return None

    s = df[group_col]

    if s.notna().sum() == 0:
        return None

    nunique = s.nunique(dropna=True)

    if nunique <= 1:
        return None

    if pd.api.types.is_numeric_dtype(s):
        n = s.notna().sum()
        if nunique > min(20, max(5, n // 3)):
            return None

    return group_col


def detect_value_cols(df, id_col, group_col, prefixes=("処方",)):
    cols = [c for c in df.columns if any(str(c).startswith(p) for p in prefixes)]

    if cols:
        return cols

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    exclude_cols = [id_col]
    if group_col:
        exclude_cols.append(group_col)

    return [c for c in numeric_cols if c not in exclude_cols]


def safe_name(s):
    s = str(s)
    return re.sub(r"[\\/:*?\"<>|]", "_", s)


# 3) FileUpload対応

def _bytes_from_uploaded_content(content):
    if content is None:
        return None

    if isinstance(content, (bytes, bytearray)):
        return bytes(content)

    if hasattr(content, "tobytes"):
        return content.tobytes()

    return bytes(content)


def normalize_uploaded_item(uploader_value):
    v = uploader_value

    if not v:
        return None

    if isinstance(v, (list, tuple)):
        return v[0]

    if isinstance(v, dict):
        return list(v.values())[0]

    raise TypeError(f"FileUpload.value の型が想定外: {type(v)}")


def save_uploaded_xlsx(uploader_value, save_as="uploaded_input.xlsx"):
    item = normalize_uploaded_item(uploader_value)

    if item is None:
        return None

    if isinstance(item, dict):
        content = item.get("content", None)
    else:
        try:
            content = item["content"]
        except Exception:
            content = getattr(item, "content", None)

    b = _bytes_from_uploaded_content(content)

    if b is None:
        raise KeyError("アップロードから content を取得できませんでした。")

    path = Path(save_as)

    with open(path, "wb") as f:
        f.write(b)

    return path


def make_output_dirs(output_root=OUTPUT_ROOT, input_stem=None):
    base_root = Path(output_root)
    base_root.mkdir(parents=True, exist_ok=True)

    if input_stem:
        folder_name = f"{safe_name(input_stem)}_analyzed"
    else:
        folder_name = "analyzed"

    root = base_root / folder_name
    excel_dir = root / "excel"
    plot_dir = root / "plots_png"
    table_dir = root / "tables"
    log_dir = root / "logs"

    for d in [root, excel_dir, plot_dir, table_dir, log_dir]:
        d.mkdir(parents=True, exist_ok=True)

    return {"root": root, "excel": excel_dir, "plots": plot_dir, "tables": table_dir, "logs": log_dir, "run_label": folder_name}


def save_table_csv(df, path):
    if df is None:
        df = pd.DataFrame()
    df = clean_df_for_excel(df)
    df.to_csv(path, index=False, encoding="utf-8-sig")


def apply_value_range_filter(df, xcol, ycol, use_range=False, lo=None, hi=None, mode="both"):
    if not use_range:
        return df.copy()

    if lo is None or hi is None:
        return df.copy()

    if lo > hi:
        lo, hi = hi, lo

    x = pd.to_numeric(df[xcol], errors="coerce")
    y = pd.to_numeric(df[ycol], errors="coerce")

    mx = pd.Series(True, index=df.index)
    my = pd.Series(True, index=df.index)

    mx &= x >= lo
    mx &= x <= hi
    my &= y >= lo
    my &= y <= hi

    if mode == "both":
        m = mx & my
    elif mode == "either":
        m = mx | my
    elif mode == "x_only":
        m = mx
    elif mode == "y_only":
        m = my
    else:
        m = mx & my

    return df.loc[m].copy()


def pearson_r(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    if len(x) < 2:
        return np.nan

    return float(np.corrcoef(x, y)[0, 1])


def regression_fit(x, y, method="OLS", deming_lambda=1.0, theilsen_max_pairs=40000, seed=0):
    x = np.asarray(x, float)
    y = np.asarray(y, float)

    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)

    if n < 2:
        return np.nan, np.nan

    if method == "OLS":
        a, b = np.polyfit(x, y, 1)
        return float(a), float(b)

    if method == "Deming":
        lx = float(deming_lambda) if deming_lambda else 1.0
        xbar = x.mean()
        ybar = y.mean()
        sxx = np.mean((x - xbar) ** 2)
        syy = np.mean((y - ybar) ** 2)
        sxy = np.mean((x - xbar) * (y - ybar))

        if sxy == 0:
            a = 0.0
        else:
            a = (syy - lx * sxx + math.sqrt((syy - lx * sxx) ** 2 + 4 * lx * sxy * sxy)) / (2 * sxy)
        b = ybar - a * xbar
        return float(a), float(b)

    if method == "TheilSen":
        rng = np.random.default_rng(seed)
        idx = np.arange(n)
        total_pairs = n * (n - 1) // 2
        slopes = []

        if total_pairs <= theilsen_max_pairs:
            for i in range(n - 1):
                dx = x[i + 1:] - x[i]
                dy = y[i + 1:] - y[i]
                mask = dx != 0
                slopes.extend((dy[mask] / dx[mask]).tolist())
        else:
            for _ in range(theilsen_max_pairs):
                i, j = rng.choice(idx, 2, replace=False)
                dx = x[j] - x[i]
                if dx == 0:
                    continue
                slopes.append((y[j] - y[i]) / dx)

        a = float(np.median(slopes)) if slopes else 0.0
        b = float(np.median(y - a * x))
        return a, b

    a, b = np.polyfit(x, y, 1)
    return float(a), float(b)


def passing_bablok_fit_with_ci(x, y, alpha=0.05, max_pairs=200000, seed=0):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    n = len(x)

    if n < 2:
        return np.nan, np.nan, {"pb_n_slopes": 0, "pb_ci_method": "n<2"}

    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    slopes = []
    for _ in range(min(max_pairs, max(1, n * (n - 1) // 2))):
        i, j = rng.choice(idx, 2, replace=False)
        dx = x[j] - x[i]
        dy = y[j] - y[i]
        if dx == 0:
            continue
        slopes.append(dy / dx)

    if not slopes:
        return np.nan, np.nan, {"pb_n_slopes": 0, "pb_ci_method": "no_slopes"}

    med = float(np.median(slopes))
    intercept = float(np.median(y - med * x))
    return med, intercept, {"pb_n_slopes": len(slopes), "pb_ci_method": "bootstrap"}


def regression_fit_info(x, y, method="OLS", deming_lambda=1.0):
    if method == "PassingBablok":
        a, b, info = passing_bablok_fit_with_ci(x, y)
        return a, b, info

    a, b = regression_fit(x, y, method=method, deming_lambda=deming_lambda)
    return a, b, {}


# ============================================================
# 7) MAD外れ値検出
# ============================================================

import numpy as np

def mad(arr):
    arr = np.asarray(arr, dtype=float)
    med = np.nanmedian(arr)
    return np.nanmedian(np.abs(arr - med))


def compute_residual_and_zmad(df, id_col, xcol, ycol, slope, intercept):
    sub = df[[id_col, xcol, ycol]].dropna().copy()

    x = sub[xcol].astype(float)
    y = sub[ycol].astype(float)

    # 予測値
    sub["pred_y"] = slope * x + intercept

    # residual
    sub["residual"] = y - sub["pred_y"]

    # MAD
    resid = sub["residual"].values
    s = mad(resid)

    scale = 1.4826 * s if (np.isfinite(s) and s > 0) else np.nan

    if np.isfinite(scale):
        sub["z_MAD"] = sub["residual"] / scale
    else:
        sub["z_MAD"] = np.nan

    sub["abs_z_MAD"] = np.abs(sub["z_MAD"])

    return sub


def classify_outlier(abs_z):
    if not np.isfinite(abs_z):
        return "not_evaluable"
    elif abs_z >= 5.0:
        return "strong_candidate"
    elif abs_z >= 3.5:
        return "candidate"
    elif abs_z >= 2.5:
        return "mild_candidate"
    else:
        return "none"


def detect_outliers(df_metrics):
    df = df_metrics.copy()

    df["outlier_level"] = df["abs_z_MAD"].apply(classify_outlier)

    flagged = df[df["outlier_level"].isin(["candidate", "strong_candidate"])].copy()

    return df, flagged


def _excel_safe_value(v):
    if v is None:
        return ""
    if isinstance(v, (float, np.floating)):
        if np.isfinite(v):
            return float(v)
        return ""
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, bool):
        return bool(v)
    if isinstance(v, (list, tuple, dict)):
        return str(v)
    return v


def clean_df_for_excel(df):
    if df is None:
        return pd.DataFrame()
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_object_dtype(out[col]):
            continue
        out[col] = out[col].astype(object)
    return out


def insert_images_into_excel(input_xlsx, output_xlsx, image_paths, plot_sheet="plots", start_cell="A1", row_step=44):
    if not image_paths:
        return

    wb = load_workbook(input_xlsx)
    ws = wb.create_sheet(title=plot_sheet)
    row = 1
    for path in image_paths:
        img = XLImage(str(path))
        ws.add_image(img, start_cell)
        row += row_step
    wb.save(output_xlsx)


def write_df_to_sheet(xlsx_path, df, sheet_name, index=False):
    df = clean_df_for_excel(df)
    wb = load_workbook(xlsx_path)
    if sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        ws.delete_rows(1, ws.max_row)
        ws.delete_cols(1, ws.max_column)
    else:
        ws = wb.create_sheet(title=sheet_name)
    ws.append([_excel_safe_value(v) for v in df.columns])
    for row in df.itertuples(index=False, name=None):
        ws.append([_excel_safe_value(v) for v in row])
    wb.save(xlsx_path)


def groupwise_stats(df, id_col, group_col, xcol, ycol, method, lam):
    if df is None or df.empty:
        return pd.DataFrame()
    if group_col is None or group_col not in df.columns:
        return pd.DataFrame()
    g = df[[id_col, group_col, xcol, ycol]].copy()
    g = g.dropna(subset=[xcol, ycol])
    if g.empty:
        return pd.DataFrame()
    out = g.groupby(group_col).agg(n=(id_col, "count"), mean_x=(xcol, "mean"), mean_y=(ycol, "mean"))
    out = out.reset_index()
    out["regression"] = method
    out["regression_label"] = REGRESSION_LABELS.get(method, method)
    return out


def compute_pair_sample_metrics(df, id_col, group_col, xcol, ycol, a, b):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df[[id_col, xcol, ycol]].copy()
    out = out.dropna(subset=[xcol, ycol])
    if out.empty:
        return pd.DataFrame()
    out["diff_y_minus_x"] = out[ycol] - out[xcol]
    out["abs_diff_yx"] = out["diff_y_minus_x"].abs()
    out["mean_xy"] = (out[xcol] + out[ycol]) / 2
    out["rel_diff_yx_pct"] = out["diff_y_minus_x"] / out[xcol] * 100
    out["bias_pct_vs_x"] = out["diff_y_minus_x"] / out[xcol] * 100
    out["ratio_y_over_x"] = out[ycol] / out[xcol]
    out["pred_y"] = a * out[xcol] + b
    out["residual"] = out[ycol] - out["pred_y"]
    out["abs_residual"] = out["residual"].abs()
    out["z_MAD"] = np.nan
    out["abs_z_MAD"] = np.nan
    out["outlier_level"] = "normal"
    out["outlier_basis"] = ""
    out["outlier_note"] = ""
    return out


def make_metric_definitions_df():
    return pd.DataFrame([
        {"metric": "r", "definition": "Pearson相関係数"},
        {"metric": "slope", "definition": "回帰直線の傾き"},
        {"metric": "intercept", "definition": "回帰直線の切片"},
        {"metric": "BA_bias(y-x)", "definition": "Bland-Altman bias"},
    ])

def get_outlier_color(sid, pair_key, metadata):
    try:
        level = metadata[sid]["outliers"][pair_key]["level"]
    except KeyError:
        return "blue"

    if level == "strong_candidate":
        return "red"
    elif level == "candidate":
        return "orange"
    elif level == "mild_candidate":
        return "yellow"
    else:
        return "blue"


help_html = widgets.HTML("<div style='border:1px solid #ccc; padding:8px;'>解析対象は parsed-data フォルダを選択してください。未選択なら最新のフォルダを自動で使います。</div>")

input_dir_dd = widgets.Dropdown(options=["(未選択)"], value="(未選択)", description="解析対象フォルダ")
refresh_dir_btn = widgets.Button(description="フォルダ候補を更新", button_style="info")
sheet_dd = widgets.Dropdown(options=["(未選択)"], value="(未選択)")
mode_dd = widgets.Dropdown(options=[("隣接ペア", "adjacent"), ("全ペア", "all"), ("基準列対比", "baseline")], value="baseline")
baseline_sel = widgets.SelectMultiple(options=[], value=(), description="基準処方", rows=8)
all_reg_ck = widgets.Checkbox(value=False, description="全回帰法で出力")
reg_dd = widgets.Dropdown(options=[("OLS（最小二乗）", "OLS"), ("Deming（両軸誤差）", "Deming"), ("Theil-Sen（ロバスト）", "TheilSen"), ("Passing-Bablok（ノンパラメトリック）", "PassingBablok")], value="PassingBablok")
deming_lambda = widgets.FloatText(value=1.0, description="λ(Deming)")
best_mode = widgets.ToggleButtons(options=[("|r|最大", "abs"), ("r最大（正相関優先）", "pos")], value="pos", description="ベスト")
z_thresh = widgets.FloatSlider(value=3.5, min=1.5, max=8.0, step=0.1, description="乖離z(MAD)")
show_outlier_label_ck = widgets.Checkbox(value=True, description="乖離IDラベル表示")
label_top = widgets.IntSlider(value=8, min=0, max=30, step=1, description="乖離IDラベル数")
use_range_ck = widgets.Checkbox(value=False, description="対象範囲で絞る")
range_min_txt = widgets.FloatText(value=0.0, description="下限")
range_max_txt = widgets.FloatText(value=100.0, description="上限")
range_mode_dd = widgets.Dropdown(options=[("XとYの両方が範囲内", "both"), ("XまたはYが範囲内", "either"), ("Xのみ範囲内", "x_only"), ("Yのみ範囲内", "y_only")], value="both", description="範囲条件")
add_diag_ck = widgets.Checkbox(value=True, description="y=x補助線")
show_fit_ck = widgets.Checkbox(value=True, description="回帰直線")
show_stats_ck = widgets.Checkbox(value=True, description="統計表示")
show_py_ck = widgets.Checkbox(value=True, description="Python上に表示")
max_show = widgets.IntSlider(value=6, min=0, max=30, step=1, description="表示上限")
show_equation_ck = widgets.Checkbox(value=True, description="回帰式を表示")
show_ci_ck = widgets.Checkbox(value=True, description="95%CIを表示")
show_ba_text_ck = widgets.Checkbox(value=True, description="BA統計を表示")
show_outlier_text_ck = widgets.Checkbox(value=True, description="乖離数を表示")
normal_point_size = widgets.IntSlider(value=16, min=4, max=80, step=2, description="通常点サイズ")
outlier_point_size = widgets.IntSlider(value=28, min=6, max=120, step=2, description="乖離○サイズ")
outlier_line_width = widgets.FloatSlider(value=1.2, min=0.5, max=4.0, step=0.1, description="乖離○線幅")
fig_width_slider = widgets.IntSlider(value=16, min=8, max=28, step=1, description="図横サイズ")
fig_height_slider = widgets.IntSlider(value=10, min=6, max=20, step=1, description="図高さ")
plot_dpi_slider = widgets.IntSlider(value=150, min=80, max=300, step=10, description="DPI")
load_btn = widgets.Button(description="列を読み込み（基準候補更新）", button_style="info")
run_btn = widgets.Button(description="実行（解析→結果出力）", button_style="success")
status = widgets.Output()
plots_out = widgets.Output()

display(widgets.VBox([widgets.HTML("<b>①フォルダ選択 → ②列読み込み → ③実行</b>"), help_html, input_dir_dd, refresh_dir_btn, widgets.HBox([mode_dd, all_reg_ck, reg_dd, deming_lambda]), widgets.HBox([baseline_sel, widgets.VBox([best_mode, z_thresh, show_outlier_label_ck, label_top])]), widgets.HBox([use_range_ck, range_min_txt, range_max_txt, range_mode_dd]), widgets.HBox([add_diag_ck, show_fit_ck, show_stats_ck, show_py_ck, max_show]), widgets.HBox([show_equation_ck, show_ci_ck, show_ba_text_ck, show_outlier_text_ck]), widgets.HBox([normal_point_size, outlier_point_size, outlier_line_width]), widgets.HBox([fig_width_slider, fig_height_slider, plot_dpi_slider]), widgets.HBox([load_btn, run_btn]), status, plots_out]))

_state = {"xlsx_path": None, "source_path": None, "source_label": None, "df": None, "id_col": None, "group_col": None, "value_cols": None}


def discover_input_dirs():
    parsed_root = resolve_project_root() / "data" / "parsed-data"
    if not parsed_root.exists():
        return []

    dirs = []
    for p in sorted(parsed_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True):
        if p.is_dir() and (p / "measurement.parquet").exists() and (p / "metadata.json").exists():
            dirs.append(p)
    return dirs


def refresh_input_dir_list():
    dirs = discover_input_dirs()
    if dirs:
        input_dir_dd.options = [(p.name, str(p)) for p in dirs]
        input_dir_dd.value = str(dirs[0])
    else:
        input_dir_dd.options = ["(未選択)"]
        input_dir_dd.value = "(未選択)"


refresh_input_dir_list()
refresh_dir_btn.on_click(lambda _=None: refresh_input_dir_list())


def refresh_sheet_list(xlsx_path):
    xl = pd.ExcelFile(xlsx_path, engine="openpyxl")
    sheets = xl.sheet_names
    if sheets:
        sheet_dd.options = sheets
        sheet_dd.value = sheets[0]
    else:
        sheet_dd.options = ["(未選択)"]
        sheet_dd.value = "(未選択)"


def set_default_range_from_data(df, value_cols):
    vals = []
    for c in value_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        vals.append(s)
    if not vals:
        return
    allv = pd.concat(vals, axis=0).dropna()
    if allv.empty:
        return
    lo0 = float(np.nanmin(allv))
    hi0 = float(np.nanmax(allv))
    if np.isfinite(lo0):
        range_min_txt.value = lo0
    if np.isfinite(hi0):
        range_max_txt.value = hi0


def load_columns(_=None):
    with status:
        clear_output()
        selected_dir = input_dir_dd.value
        target_dir = Path(selected_dir) if selected_dir and selected_dir != "(未選択)" else None

        if target_dir is not None:
            try:
                df, _, _, parsed_dir, id_col, group_col, value_cols = load_parsed_data_for_notebook(target_dir)
            except Exception as e:
                print("解析対象フォルダの読み込みに失敗しました。", repr(e))
                return
            _state.update({"xlsx_path": None, "source_path": parsed_dir, "source_label": parsed_dir.name, "df": df, "id_col": id_col, "group_col": group_col, "value_cols": value_cols})
            baseline_sel.options = value_cols
            if value_cols:
                baseline_sel.value = (value_cols[0],)
            set_default_range_from_data(df, value_cols)
            print("解析対象フォルダを読み込みました")
            print("  入力元 :", parsed_dir)
            print("  ID列   :", id_col)
            print("  種別列 :", group_col if group_col else "(なし)")
            print("  比較列 :", value_cols)
            print(f"  対象範囲初期値: {range_min_txt.value:.4g} ～ {range_max_txt.value:.4g}")
            if len(value_cols) < 2:
                print("注意：比較列が2列未満です。解析には2列以上必要です。")
            return

        try:
            df, _, _, parsed_dir, id_col, group_col, value_cols = load_parsed_data_for_notebook()
        except Exception as e:
            print("解析対象データの読み込みに失敗しました。", repr(e))
            return

        _state.update({"xlsx_path": None, "source_path": parsed_dir, "source_label": parsed_dir.name, "df": df, "id_col": id_col, "group_col": group_col, "value_cols": value_cols})
        baseline_sel.options = value_cols
        if value_cols:
            baseline_sel.value = (value_cols[0],)
        set_default_range_from_data(df, value_cols)
        print("解析入力を自動読み込みしました")
        print("  入力元 :", parsed_dir)
        print("  ID列   :", id_col)
        print("  種別列 :", group_col if group_col else "(なし)")
        print("  比較列 :", value_cols)
        print(f"  対象範囲初期値: {range_min_txt.value:.4g} ～ {range_max_txt.value:.4g}")
        if len(value_cols) < 2:
            print("注意：比較列が2列未満です。解析には2列以上必要です。")

load_btn.on_click(load_columns)


def run_all(_=None):
    with status:
        clear_output()
    with plots_out:
        clear_output()

    if _state.get("df") is None:
        load_columns()

    xlsx_path = _state.get("xlsx_path")
    source_path = _state.get("source_path")
    source_label = _state.get("source_label")
    df = _state.get("df")
    id_col = _state.get("id_col")
    group_col = normalize_group_col(df, _state.get("group_col"))
    value_cols = _state.get("value_cols")

    if df is None or id_col is None or not value_cols or len(value_cols) < 2:
        with status:
            print("解析に必要な列が不足しています。")
            print("検体ID列と、2列以上の比較列が必要です。")
        return

    if mode_dd.value == "adjacent":
        pairs = list(zip(value_cols[:-1], value_cols[1:]))
    elif mode_dd.value == "all":
        pairs = list(itertools.combinations(value_cols, 2))
    else:
        bases = list(baseline_sel.value) if baseline_sel.value else [value_cols[0]]
        pairs = []
        for b0 in bases:
            for c in value_cols:
                if c != b0:
                    pairs.append((b0, c))

    methods = REGRESSION_METHODS_ALL if all_reg_ck.value else [reg_dd.value]
    lam = deming_lambda.value
    lo_range = float(range_min_txt.value)
    hi_range = float(range_max_txt.value)
    if lo_range > hi_range:
        lo_range, hi_range = hi_range, lo_range

    try:
        dirs = make_output_dirs(OUTPUT_ROOT, input_stem=(source_label or (xlsx_path.stem if xlsx_path is not None else "parsed_input")))
    except Exception as e:
        with status:
            print("出力先フォルダ作成エラー:", repr(e))
        return

    out_dir = dirs["plots"]
    excel_dir = dirs["excel"]
    summary_rows = []
    group_rows = []
    outlier_tables = []
    sample_metric_tables = []
    img_paths = []
    shown = 0

    # =============================
    # metadata読み込み
    # =============================
    from pathlib import Path
    import json

    metadata_path = Path(dirs["root"]) / "metadata.json"

    if metadata_path.exists():
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)
    else:
        metadata = {}

    for method in methods:
        for xcol, ycol in pairs:
            # df_pair = apply_value_range_filter(df, xcol=xcol, ycol=ycol, use_range=use_range_ck.value, lo=lo_range, hi=hi_range, mode=range_mode_dd.value)
            df_pair = df[[id_col, xcol, ycol]].copy()
            
            df_pair[xcol] = pd.to_numeric(df_pair[xcol], errors="coerce")
            df_pair[ycol] = pd.to_numeric(df_pair[ycol], errors="coerce")

            df_pair = df_pair.dropna(subset=[xcol, ycol])

            sub = df_pair[[xcol, ycol]]
            if len(sub) < 2:
                continue

            x = pd.to_numeric(sub[xcol], errors="coerce").astype(float).to_numpy()
            y = pd.to_numeric(sub[ycol], errors="coerce").astype(float).to_numpy()

            r = pearson_r(x, y)
            a, b, fit_info = regression_fit_info(x, y, method=method, deming_lambda=lam)

            # ===== MAD計算 =====
            df_metrics = compute_residual_and_zmad(
                df=df_pair,
                id_col=id_col,
                xcol=xcol,
                ycol=ycol,
                slope=a,
                intercept=b
            )

            df_metrics, flagged = detect_outliers(df_metrics)

            # df_pairを置き換える
            df_pair = df_metrics.copy()

            pair_key = f"{xcol}_vs_{ycol}"

            print("after MAD:", len(df_metrics))
            print("after assign df_pair:", len(df_pair))


            # ====================================
            # metadata書き込み
            # ====================================

            for _, row in df_pair.iterrows():
                sid = str(row[id_col])

                if sid not in metadata:
                    metadata[sid] = {}

                if "outliers" not in metadata[sid]:
                    metadata[sid]["outliers"] = {}

                level = row.get("outlier_level", None)
                z_mad = row.get("z_MAD", np.nan)

                metadata[sid]["outliers"][pair_key] = {
                    "level": level,
                    "z_MAD": float(z_mad) if np.isfinite(z_mad) else None
                }

            # 保存（1回でOK）
            with open(metadata_path, "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=2, ensure_ascii=False)

            # ====================================
            # color_list（metadata AFTER）
            # ====================================

            sub = df_pair[[id_col, xcol, ycol]].dropna()

            color_list = []
            for _, row in sub.iterrows():
                sid = str(row[id_col])
                color = get_outlier_color(sid, pair_key, metadata)
                color_list.append(color)

            # 念のため安全化
            if len(color_list) != len(sub):
                color_list = ["blue"] * len(sub)

            # ====================================
            # metadata.jsonへの書き込み（修正版）
            # ====================================

            from pathlib import Path
            import json
            import numpy as np

            metadata_path = Path(dirs["root"]) / "metadata.json"

            # 既存metadata読み込み
            if metadata_path.exists():
                with open(metadata_path, "r", encoding="utf-8") as f:
                    metadata = json.load(f)
            else:
                metadata = {}

            pair_key = f"{xcol}_vs_{ycol}"

            for _, row in df_pair.iterrows():
                sid = str(row[id_col])

                if sid not in metadata:
                    metadata[sid] = {}

                if "outliers" not in metadata:
                    metadata[sid]["outliers"] = {}

                # 安全に値取得
                level = row["outlier_level"] if "outlier_level" in row else None
                z_mad = row["z_MAD"] if "z_MAD" in row else np.nan

                metadata[sid]["outliers"][pair_key] = {
                    "level": level,
                    "z_MAD": float(z_mad) if np.isfinite(z_mad) else None
                }

            # 保存
            with open(metadata_path, "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=2, ensure_ascii=False)


            fig, used_sub, _, bias, loa = plot_suite(
                df=df_pair,
                id_col=id_col,
                group_col=group_col,
                xcol=xcol, ycol=ycol,
                method=method,
                lam=lam,
                a=a,
                b=b,
                r=r,
                fit_info=fit_info,
                z_thresh=z_thresh.value,
                show_outlier_labels=show_outlier_label_ck.value,
                outlier_label_top=label_top.value,
                add_diag=add_diag_ck.value,
                show_fit=show_fit_ck.value,
                show_stats=show_stats_ck.value,
                show_equation=show_equation_ck.value,
                show_ci=show_ci_ck.value,
                show_ba_text=show_ba_text_ck.value,
                show_outlier_text=show_outlier_text_ck.value,
                normal_s=normal_point_size.value,
                outlier_s=outlier_point_size.value,
                outlier_lw=outlier_line_width.value,
                fig_width=fig_width_slider.value,
                fig_height=fig_height_slider.value,
                dpi=plot_dpi_slider.value,
                external_colors=color_list
                )
            
            # ✅ external_colorsを安全化
            if color_list is None or len(color_list) != len(df_pair):
                color_list = ["blue"] * len(df_pair)
            if fig is None:
                continue
            if show_py_ck.value and max_show.value > 0 and shown < max_show.value:
                with plots_out:
                    display(fig)
                shown += 1

            png = out_dir / f"QC_{safe_name(method)}_{safe_name(ycol)}_vs_{safe_name(xcol)}.png"
            fig.savefig(png, bbox_inches="tight")
            plt.close(fig)
            img_paths.append(png)

            n_out = int(len(flagged)) if flagged is not None else 0
            summary_rows.append({
                "regression": method,
                "regression_label": REGRESSION_LABELS.get(method, method),
                "X": xcol,
                "Y": ycol,
                "n": len(x),
                "r": r,
                "|r|": abs(r) if np.isfinite(r) else np.nan,
                "slope": a,
                "intercept": b,
                "lambda": lam if method == "Deming" else np.nan,
                "slope_95CI_low": fit_info.get("slope_ci_low", np.nan),
                "slope_95CI_high": fit_info.get("slope_ci_high", np.nan),
                "intercept_95CI_low": fit_info.get("intercept_ci_low", np.nan),
                "intercept_95CI_high": fit_info.get("intercept_ci_high", np.nan),
                "PB_n_slopes": fit_info.get("pb_n_slopes", np.nan),
                "PB_CI_method": fit_info.get("pb_ci_method", ""),
                "BA_bias(y-x)": bias,
                "BA_LoA_low": loa[0],
                "BA_LoA_high": loa[1],
                "n_outliers": n_out,
                "outlier_z_thresh": z_thresh.value,
                "outlier_basis": "回帰残差 y-(slope*x+intercept) をMADで標準化",
                "outlier_note": "乖離候補であり、自動除外ではない",
                "outlier_excluded_from_fit": False,
                "fit_data_note": "相関係数・回帰係数は乖離候補を除外せず、解析対象データ全体で算出",
                "range_filter_used": bool(use_range_ck.value),
                "range_min": lo_range if use_range_ck.value else np.nan,
                "range_max": hi_range if use_range_ck.value else np.nan,
                "range_condition": range_mode_dd.value if use_range_ck.value else "",
                "normal_point_size": normal_point_size.value,
                "outlier_point_size": outlier_point_size.value,
                "outlier_line_width": outlier_line_width.value,
                "figure_width": fig_width_slider.value,
                "figure_height": fig_height_slider.value,
                "plot_dpi": plot_dpi_slider.value,
                "output_root": str(dirs["root"]),
            })

            gdf = groupwise_stats(df_pair, id_col, group_col, xcol, ycol, method, lam)
            if not gdf.empty:
                group_rows.append(gdf)

            if flagged is not None and not flagged.empty:
                outlier_tables.append(flagged.assign(regression=method, regression_label=REGRESSION_LABELS.get(method, method), slope=a, intercept=b, r=r))

            metrics_df = compute_pair_sample_metrics(df=df_pair, id_col=id_col, group_col=group_col, xcol=xcol, ycol=ycol, a=a, b=b)
            if not metrics_df.empty:
                metrics_df["regression"] = method
                metrics_df["regression_label"] = REGRESSION_LABELS.get(method, method)
                metrics_df["slope"] = a
                metrics_df["intercept"] = b
                metrics_df["r"] = r
                metrics_df["slope_95CI_low"] = fit_info.get("slope_ci_low", np.nan)
                metrics_df["slope_95CI_high"] = fit_info.get("slope_ci_high", np.nan)
                metrics_df["intercept_95CI_low"] = fit_info.get("intercept_ci_low", np.nan)
                metrics_df["intercept_95CI_high"] = fit_info.get("intercept_ci_high", np.nan)
                metrics_df["outlier_z_thresh"] = z_thresh.value
                metrics_df["outlier_excluded_from_fit"] = False
                metrics_df["fit_data_note"] = "相関係数・回帰係数は乖離候補を除外せず、解析対象データ全体で算出"
                metrics_df["range_filter_used"] = bool(use_range_ck.value)
                metrics_df["range_min"] = lo_range if use_range_ck.value else np.nan
                metrics_df["range_max"] = hi_range if use_range_ck.value else np.nan
                metrics_df["range_condition"] = range_mode_dd.value if use_range_ck.value else ""
                sample_metric_tables.append(metrics_df)

    if not summary_rows:
        with status:
            print("作図・解析できるペアがありませんでした。")
            print("欠損が多い、対象範囲が狭すぎる、または比較列が不足している可能性があります。")
            print("=== DEBUG START ===")
            print("df_pair shape:", df_pair.shape)
            print("columns:", df_pair.columns.tolist())
            print("color_list length:", len(color_list) if color_list is not None else None)
            print("df_pair length:", len(df_pair))
            print("z_MAD exists:", "z_MAD" in df_pair.columns)
            print("outlier_level exists:", "outlier_level" in df_pair.columns)
            print("xcol:", xcol)
            print("ycol:", ycol)

            print("before dropna:", len(df_pair))
            print("after dropna:", len(df_pair[[xcol, ycol]].dropna()))

            print(df_pair[[xcol, ycol]].head(10))

            print(len(df_pair))
            print(df_pair[[xcol, ycol]].head())

            print("after MAD:", len(df_metrics))
            print("after assign df_pair:", len(df_pair))

            print("id_col:", id_col)
            print(measurement_df.columns)

            print("=== DEBUG END ===")
        return

    summary_df = pd.DataFrame(summary_rows)
    if best_mode.value == "abs":
        best = summary_df.sort_values("|r|", ascending=False).iloc[0]
        best_label = "|r|最大"
    else:
        best = summary_df.sort_values("r", ascending=False).iloc[0]
        best_label = "r最大（正相関優先）"

    with status:
        print("=== 実行結果 ===")
        print("入力:", xlsx_path if xlsx_path is not None else source_path)
        print("入力種別:", "Excel" if xlsx_path is not None else "parquet/metadata")
        print("出力root:", dirs["root"])
        print("モード:", mode_dd.value, " / ペア数:", len(pairs))
        print("回帰法:", "全回帰法" if all_reg_ck.value else REGRESSION_LABELS.get(reg_dd.value, reg_dd.value))
        print(f"ベスト相関（{best_label}）: {best['regression_label']} / {best['Y']} vs {best['X']}")
        print(f"  n={int(best['n'])}  r={best['r']:.4f}  slope={best['slope']:.4f}  intercept={best['intercept']:.4f}")
        print(f"  BA bias={best['BA_bias(y-x)']:.4g}  LoA=[{best['BA_LoA_low']:.4g}, {best['BA_LoA_high']:.4g}]  outliers={int(best['n_outliers'])}")
        print("※相関係数・回帰係数は、乖離候補を除外せずに算出しています。")
        print(f"Python表示枚数: {shown}")

    if xlsx_path is not None:
        output_xlsx = excel_dir / f"{xlsx_path.stem}{OUT_SUFFIX}{xlsx_path.suffix}"
        insert_images_into_excel(input_xlsx=xlsx_path, output_xlsx=output_xlsx, image_paths=img_paths, plot_sheet=SHEET_PLOTS, start_cell="A1", row_step=44)
    else:
        output_xlsx = excel_dir / f"{safe_name(source_label or 'parsed_input')}{OUT_SUFFIX}.xlsx"
        wb = Workbook()
        # wb.remove(wb.active)
        wb.save(output_xlsx)

    summary_to_write = summary_df.drop(columns=["|r|"], errors="ignore")
    write_df_to_sheet(output_xlsx, summary_to_write, SHEET_SUMMARY)
    save_table_csv(summary_to_write, dirs["tables"] / "summary.csv")

    if group_rows:
        gs = pd.concat(group_rows, axis=0, ignore_index=True)
        write_df_to_sheet(output_xlsx, gs, SHEET_GROUP_STATS)
        save_table_csv(gs, dirs["tables"] / "group_stats.csv")
    else:
        empty_gs = pd.DataFrame([{"note": "有効な種別列がない、または群別統計を作成できるデータがありません。"}])
        write_df_to_sheet(output_xlsx, empty_gs, SHEET_GROUP_STATS)
        save_table_csv(empty_gs, dirs["tables"] / "group_stats.csv")

    if outlier_tables:
        odf = pd.concat(outlier_tables, axis=0, ignore_index=True)
        cols_first = ["regression", "regression_label", "X", "Y", id_col]
        if group_col:
            cols_first.append(group_col)
        cols_first += ["slope", "intercept", "r", "diff_y_minus_x", "abs_diff_yx", "mean_xy", "rel_diff_yx_pct", "bias_pct_vs_x", "ratio_y_over_x", "pred_y", "residual", "abs_residual", "z_MAD", "abs_z_MAD", "outlier_level", "outlier_basis", "outlier_note"]
        cols_first = [c for c in cols_first if c in odf.columns]
        rest_cols = [c for c in odf.columns if c not in cols_first]
        odf = odf[cols_first + rest_cols]
        write_df_to_sheet(output_xlsx, odf, SHEET_OUTLIERS)
        save_table_csv(odf, dirs["tables"] / "outliers.csv")
    else:
        empty_out = pd.DataFrame([{"note": "指定した zMAD 閾値以上の乖離候補はありませんでした。"}])
        write_df_to_sheet(output_xlsx, empty_out, SHEET_OUTLIERS)
        save_table_csv(empty_out, dirs["tables"] / "outliers.csv")

    if sample_metric_tables:
        smdf = pd.concat(sample_metric_tables, axis=0, ignore_index=True)
        cols_first = ["regression", "regression_label", "X", "Y", id_col]
        if group_col:
            cols_first.append(group_col)
        metric_cols = ["slope", "intercept", "r", "slope_95CI_low", "slope_95CI_high", "intercept_95CI_low", "intercept_95CI_high", "diff_y_minus_x", "abs_diff_yx", "mean_xy", "rel_diff_yx_pct", "bias_pct_vs_x", "ratio_y_over_x", "pred_y", "residual", "abs_residual", "z_MAD", "abs_z_MAD", "outlier_level", "outlier_z_thresh", "outlier_basis", "outlier_note", "outlier_excluded_from_fit", "fit_data_note", "range_filter_used", "range_min", "range_max", "range_condition"]
        cols_first += metric_cols
        cols_first = [c for c in cols_first if c in smdf.columns]
        rest_cols = [c for c in smdf.columns if c not in cols_first]
        smdf = smdf[cols_first + rest_cols]
        write_df_to_sheet(output_xlsx, smdf, SHEET_SAMPLE_METRICS)
        save_table_csv(smdf, dirs["tables"] / "sample_metrics.csv")
    else:
        empty_smdf = pd.DataFrame([{"note": "sample_metricsを作成できるデータがありません。"}])
        save_table_csv(empty_smdf, dirs["tables"] / "sample_metrics.csv")

    metric_def_df = make_metric_definitions_df()
    write_df_to_sheet(output_xlsx, metric_def_df, SHEET_METRIC_DEFINITIONS)
    save_table_csv(metric_def_df, dirs["tables"] / "metric_definitions.csv")

    log_text = f"""実行ログ

入力ファイル:
{xlsx_path if xlsx_path is not None else source_path}

入力シート:
{sheet_dd.value if xlsx_path is not None else '(parsed-data)'}

出力フォルダ:
{dirs['root']}

出力Excel:
{output_xlsx}

モード:
{mode_dd.value}

ペア数:
{len(pairs)}

回帰法:
{"全回帰法" if all_reg_ck.value else REGRESSION_LABELS.get(reg_dd.value, reg_dd.value)}

Deming lambda:
{lam}

ベスト判定:
{best_label}

ベスト:
{best['regression_label']} / {best['Y']} vs {best['X']}

ベスト n:
{int(best['n'])}

ベスト r:
{best['r']}

ベスト slope:
{best['slope']}

ベスト intercept:
{best['intercept']}

対象範囲フィルタ:
{bool(use_range_ck.value)}

対象範囲:
{lo_range} ～ {hi_range}

範囲条件:
{range_mode_dd.value if use_range_ck.value else ''}

乖離zMAD閾値:
{z_thresh.value}

乖離IDラベル表示:
{bool(show_outlier_label_ck.value)}

乖離IDラベル数:
{label_top.value}

95%CI表示:
{bool(show_ci_ck.value)}
※95%CI表示はグラフ内テキスト表示のON/OFFです。CI線・CI帯は描画していません。

通常点サイズ:
{normal_point_size.value}

乖離○サイズ:
{outlier_point_size.value}

乖離○線幅:
{outlier_line_width.value}

図サイズ:
{fig_width_slider.value} x {fig_height_slider.value}

DPI:
{plot_dpi_slider.value}

Python表示枚数:
{shown}

備考:
相関係数・回帰係数は、乖離候補を除外せず、欠損除外・対象範囲フィルタ後の全データで算出しています。
"""

    with open(dirs["logs"] / "run_log.txt", "w", encoding="utf-8") as f:
        f.write(log_text)

    with status:
        print("\n完了！")
        print("出力Excel:", output_xlsx)
        print("画像フォルダ:", dirs["plots"])
        print("表フォルダ:", dirs["tables"])
        print("ログフォルダ:", dirs["logs"])
        print(f"シート: {SHEET_PLOTS}, {SHEET_SUMMARY}, {SHEET_GROUP_STATS}, {SHEET_OUTLIERS}, {SHEET_SAMPLE_METRICS}, {SHEET_METRIC_DEFINITIONS}")
        print("※Excelのシート構成・主要内容は従来どおりです。")
        print("※tables/logsは補助出力です。Excel内の解析結果を壊しません。")
        print("※乖離候補は自動除外していません。")
        print("※r/slope/interceptは、欠損除外・対象範囲フィルタ後の全データで算出しています。")


run_btn.on_click(run_all)


In [15]:
print("==== DEBUG TEST ====")
print(type(df_pair))
print(df_pair.shape)
print(df_pair.head())

try:
    print("color_list length:", len(color_list))
except:
    print("color_list 未定義")

print("==== END ====")

==== DEBUG TEST ====


NameError: name 'df_pair' is not defined

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

# =========================
# データ読み込み
# =========================
df_profile = pd.read_parquet(Path(dirs["root"]) / "profile.parquet")

with open(Path(dirs["root"]) / "metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

pair_key = f"{xcol}_vs_{ycol}"

# =========================
# 描画
# =========================
fig, ax = plt.subplots(figsize=(10, 6))

for sid, gdf in df_profile.groupby("依頼No."):   # ←ここが重要

    sid = str(sid)

    # 色決定（超シンプル版）
    try:
        level = metadata[sid]["outliers"][pair_key]["level"]
    except KeyError:
        level = "none"

    if level == "strong_candidate":
        color = "red"
        lw = 2.5
    elif level == "candidate":
        color = "orange"
        lw = 2.0
    else:
        color = "blue"
        lw = 1.0

    ax.plot(
        gdf["時間"],
        gdf["吸光度"],
        color=color,
        linewidth=lw,
        alpha=0.8
    )

# =========================
# 仕上げ
# =========================
ax.set_title(f"Time Course ({pair_key})")
ax.set_xlabel("時間")
ax.set_ylabel("吸光度")

ax.grid(True, alpha=0.3)

plt.show()

NameError: name 'dirs' is not defined